# 04 - Semantic Feature Extraction

## Prerequisites
- `01_face_detection_extraction.ipynb` must be completed
- `preprocessed_faces_integrated/` must contain extracted face images

## Run Order
This notebook can run **in parallel** with `02_spatial` and `03_frequency`.

## Force Re-extraction
Set `FORCE_SEMANTIC = True` below **only** when switching semantic models
(e.g., DINOv2 to CLIP). This deletes all existing semantic features and
re-extracts them. Keep `False` for normal runs.

## Output
- `extracted_features_integrated/semantic/real|fake/.../frame_N.npy` (768-dim per face)

In [ ]:
!pip install --user timm

In [ ]:
import os
import shutil
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

import site, sys
sys.path.append(site.getusersitepackages())
import timm
import timm.data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class Config:
    BASE_DIR = Path(".")
    PREPROCESSED_DIR = BASE_DIR / "preprocessed_faces_integrated"
    FEATURES_DIR = BASE_DIR / "extracted_features_integrated"

config = Config()
(config.FEATURES_DIR / "semantic" / "real").mkdir(parents=True, exist_ok=True)
(config.FEATURES_DIR / "semantic" / "fake").mkdir(parents=True, exist_ok=True)
print("Directories ready")

In [ ]:
# ============================================================================
# FORCE RE-EXTRACTION OPTION
# ============================================================================
# Set to True ONLY when switching semantic models (e.g., DINOv2 -> CLIP).
# This deletes ALL existing semantic features and re-extracts from scratch.
# Keep False for normal runs (will skip already-extracted features).

FORCE_SEMANTIC = False

if FORCE_SEMANTIC:
    semantic_dir = config.FEATURES_DIR / "semantic"
    if semantic_dir.exists():
        old_count = sum(1 for _ in semantic_dir.rglob("*.npy"))
        shutil.rmtree(semantic_dir)
        (semantic_dir / "real").mkdir(parents=True, exist_ok=True)
        (semantic_dir / "fake").mkdir(parents=True, exist_ok=True)
        print(f"Cleared {old_count} old semantic features (force re-extraction)")
    else:
        print("No existing semantic features to clear")
else:
    print("Normal mode: will skip already-extracted features")

In [ ]:
class SemanticExtractor(nn.Module):
    """CLIP ViT-B/32 semantic feature extractor (768-dim)."""

    def __init__(self):
        super().__init__()
        self.model = timm.create_model(
            "vit_base_patch32_clip_224.openai", pretrained=True, num_classes=0
        )
        self.model.eval()

        data_config = timm.data.resolve_model_data_config(self.model)
        self.transform = timm.data.create_transform(**data_config, is_training=False)

    @torch.no_grad()
    def forward(self, img_path):
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img).unsqueeze(0).to(device)
        return self.model(img_tensor).squeeze().cpu().numpy()

print("SemanticExtractor (CLIP ViT-B/32, 768-dim) defined")

In [ ]:
def extract_single_domain(extractor, extract_fn, domain, preprocessed_dir, output_dir):
    """Extract features for one domain from all preprocessed faces.
    Skips already-extracted features for resumability."""
    image_paths = []
    for label in ["real", "fake"]:
        label_dir = preprocessed_dir / label
        if label_dir.exists():
            for video_dir in label_dir.iterdir():
                if video_dir.is_dir():
                    for img_path in video_dir.glob("*.png"):
                        image_paths.append(img_path)

    print(f"Found {len(image_paths)} face images to process")
    if not image_paths:
        print("ERROR: No images found. Run 01_face_detection_extraction.ipynb first!")
        return 0, 0, 0

    processed = 0
    skipped = 0
    errors = 0

    for img_path in tqdm(image_paths, desc=f"Extracting {domain}"):
        relative_path = img_path.relative_to(preprocessed_dir)
        feature_path = output_dir / domain / relative_path.with_suffix(".npy")

        if feature_path.exists():
            skipped += 1
            continue

        feature_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            features = extract_fn(extractor, img_path)
            np.save(feature_path, features)
            processed += 1
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  Error on {img_path.name}: {e}")

    print(f"\n{domain.upper()} complete: Processed={processed}, Skipped={skipped}, Errors={errors}")
    return processed, skipped, errors

In [ ]:
# Initialize and run
extractor = SemanticExtractor().to(device)

processed, skipped, errors = extract_single_domain(
    extractor,
    lambda e, p: e(p),
    "semantic",
    config.PREPROCESSED_DIR,
    config.FEATURES_DIR,
)

del extractor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Validation

In [ ]:
# ============================================================================
# QUICK VALIDATION
# ============================================================================

print(f"\n{'='*70}")
print("VALIDATION - Semantic Features")
print(f"{'='*70}")

total = 0
for label in ["real", "fake"]:
    feat_dir = config.FEATURES_DIR / "semantic" / label
    if feat_dir.exists():
        count = sum(
            sum(1 for f in os.listdir(d) if f.endswith(".npy"))
            for d in feat_dir.iterdir() if d.is_dir()
        )
        print(f"  {label}: {count} feature files")
        total += count
print(f"  TOTAL: {total} semantic features")

if total > 0:
    sample = next((config.FEATURES_DIR / "semantic").rglob("*.npy"))
    shape = np.load(sample).shape
    print(f"  Sample shape: {shape} (expected: (768,))")
    print(f"\nSemantic extraction successful!")
    print("Next: Ensure 02 and 03 are also complete, then run 05.")
else:
    print(f"\nNo semantic features found. Check logs above.")